In [164]:
from pathlib import Path


def read_graph(filepath):
    filepath = Path(filepath)

    n = None
    source = None
    sink = None
    edges = []

    with filepath.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("c"):
                continue

            parts = line.split()

            if parts[0] == "p":
                n = int(parts[2])

            elif parts[0] == "n":
                node = int(parts[1]) - 1
                if parts[2] == "s":
                    source = node
                elif parts[2] == "t":
                    sink = node

            elif parts[0] == "a":
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                cap = int(parts[3])
                edges.append((u, v, cap))

    # пропускная способность
    edges.sort()
    capacity = [[0] * n for _ in range(n)]
    for u, v, cap in edges:
        capacity[u][v] += cap
    return capacity, source, sink, edges

In [165]:
from collections import deque


def bfs(capacity: list, source: int, sink: int):
    n = len(capacity)
    parent = [-1] * n
    parent[source] = source

    q = deque([source])
    while q:
        v = q.popleft()
        for u in range(n):
            if parent[u] == -1 and capacity[v][u] > 0:
                parent[u] = v
                if u == sink:
                    return parent
                q.append(u)
    return None

In [166]:
import time

INF = 10**18

def edmonds_karp(capacity: list, source: int, sink: int):
    flow = 0
    iterations = 0

    while parent := bfs(capacity, source, sink):
        iterations += 1
        # find max flow for the shortest path
        cur = sink
        mn_flow = INF
        while cur != source:
            prev = parent[cur]
            mn_flow = min(mn_flow, capacity[prev][cur])
            cur = prev
        # update path capacity
        cur = sink
        while cur != source:
            prev = parent[cur]
            capacity[prev][cur] -= mn_flow
            capacity[cur][prev] += mn_flow
            cur = prev
        flow += mn_flow
    return flow, iterations


def extract_flows(capacity: list, edges: list):
    flows = []

    for u, v, cap in edges:
        flow = cap - capacity[u][v]
        if flow > 0:
            flows.append((u, v, flow))

    return flows


def run(filepath: str):
    capacity, source, sink, edges = read_graph(filepath)
    start = time.perf_counter()
    max_flow, iterations = edmonds_karp(capacity, source, sink)
    end = time.perf_counter()

    flows = extract_flows(capacity, edges)

    print(f"File: {Path(filepath).name}")
    print("-" * 40)
    print(f"Max flow: {max_flow}")
    print(f"Iterations: {iterations}")
    print(f"Time: {end - start:.6f} sec")
    print("Flows on edges:")
    for u, v, f in flows:
        print(f"{u} -> {v}: {f}")
    print("-" * 40, "\n")

In [167]:
def main():
    inputs = Path("inputs/lab03")
    for filename in inputs.iterdir():
        run(filename)

main()

File: qpbo_problem_1.cleanrescale.bq.max
----------------------------------------
Max flow: 6812
Iterations: 31
Time: 0.002138 sec
Flows on edges:
0 -> 2: 831
0 -> 3: 93
0 -> 5: 2345
0 -> 8: 400
0 -> 11: 736
0 -> 18: 354
0 -> 23: 290
0 -> 25: 411
0 -> 28: 1352
2 -> 9: 1874
2 -> 26: 162
3 -> 25: 731
4 -> 1: 290
4 -> 2: 125
4 -> 7: 208
5 -> 2: 869
5 -> 4: 623
5 -> 13: 853
6 -> 1: 411
6 -> 22: 771
7 -> 3: 46
7 -> 21: 162
8 -> 12: 1191
9 -> 1: 1352
9 -> 10: 522
10 -> 6: 211
10 -> 37: 440
11 -> 16: 971
12 -> 10: 67
12 -> 18: 1124
13 -> 10: 62
13 -> 14: 791
14 -> 8: 791
16 -> 6: 971
18 -> 11: 235
18 -> 20: 803
18 -> 29: 440
20 -> 2: 211
20 -> 3: 592
21 -> 1: 831
21 -> 24: 869
21 -> 39: 357
22 -> 1: 93
22 -> 26: 232
22 -> 39: 446
23 -> 21: 61
23 -> 24: 623
24 -> 1: 2345
25 -> 29: 171
25 -> 35: 971
26 -> 23: 394
27 -> 1: 400
27 -> 33: 791
28 -> 21: 1834
29 -> 28: 482
29 -> 31: 67
29 -> 32: 62
30 -> 1: 736
30 -> 37: 235
31 -> 27: 1191
32 -> 24: 853
33 -> 32: 791
35 -> 30: 971
37 -> 1: 354
37 ->